# 03. train / test 분할

전처리한 `user_artist_plays.csv`를 사용자별로 train 80%, test 20%로 나눈다.
나눈 파일은 협업 필터링, 콘텐츠 기반 노트북에서 똑같이 사용한다.

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from evaluate import evaluate

plays = pd.read_csv("outputs/user_artist_plays.csv")
print(plays.shape)
plays.head()

(281652, 4)


,user_id,artist_name,plays,preference
0,user_000001,4hero,146,4.990433
1,user_000001,a tribe called quest,4,1.609438
2,user_000001,adrian sherwood,1,0.693147
3,user_000001,akira kosemura,6,1.945910
4,user_000001,alif tree,15,2.772589


## 사용자별로 80:20 분할

전체를 한 번에 나누면 test가 거의 없는 사용자가 생길 수 있어서 사용자마다 따로 나눴다.

- `train_test_split(data, test_size=0.2, random_state=42)` (sklearn)
  - `test_size=0.2`: 20%를 test로 사용
  - `random_state=42`: 랜덤 결과를 고정해서 팀원 모두 같은 결과가 나오게 함

In [5]:
train_list = []
test_list = []

for user_id, group in plays.groupby("user_id"):
    tr, te = train_test_split(group, test_size=0.2, random_state=42)
    train_list.append(tr)
    test_list.append(te)

train = pd.concat(train_list)
test = pd.concat(test_list)

print("train:", len(train))
print("test:", len(test))

train: 224925
test: 56727


## 확인

test에만 있고 train에는 없는 아티스트는 어떤 모델도 추천할 수 없다. 몇 개나 되는지 확인해 봤다.

In [6]:
print("사용자 수:", train["user_id"].nunique(), test["user_id"].nunique())
print("사용자당 test 개수 평균:", round(test.groupby("user_id").size().mean(), 1))

test_only = set(test["artist_name"]) - set(train["artist_name"])
print("test에만 있는 아티스트 수:", len(test_only))

사용자 수: 973 973
사용자당 test 개수 평균: 58.3
test에만 있는 아티스트 수: 0


In [7]:
train.to_csv("outputs/train.csv", index=False)
test.to_csv("outputs/test.csv", index=False)

## 평가 함수 확인 (인기순 추천)

평가 함수가 잘 되는지 보려고 인기순 추천을 먼저 해봤다. 이게 기준선이 된다.
train에서 많은 사용자가 들은 아티스트 순서대로, 아직 안 들은 아티스트 10개를 추천한다.

사용자당 test 아티스트가 50개 이상이라 10개를 다 맞혀도 Recall@10은 낮게 나온다.

In [8]:
# 아티스트별로 들은 사용자 수
popular = train.groupby("artist_name")["user_id"].count().sort_values(ascending=False)
popular_artists = list(popular.index)

rows = []
for user_id, group in train.groupby("user_id"):
    listened = set(group["artist_name"])
    rank = 1
    for artist in popular_artists:
        if artist not in listened:
            rows.append([user_id, rank, artist])
            rank += 1
        if rank > 10:
            break

recs_popular = pd.DataFrame(rows, columns=["user_id", "rank", "artist_name"])
recs_popular.to_csv("outputs/recs_popular.csv", index=False)

evaluate(recs_popular, test, k=10)

Precision@10: 0.2953
Recall@10: 0.0546


(0.29527235354573483, 0.05463210098537418)